# Dynamic Programming Case Study

## Dynamic programming solution for the 3x3 grid-world case study.

End-to-end story of this file:
1. Import the gridworld environment and a helper to print policies.
2. `policy_evaluation()` computes state values $V^\pi$ for a fixed policy $\pi$,
   taking expectation over both:
   - action probabilities $\pi(a|s)$
   - transition probabilities $P(s', r | s, a)$
3. `compute_q_from_v()` converts those state values into action values $Q^\pi$.
4. `policy_improvement()` converts those action values into an epsilon-greedy
   policy, producing a better policy candidate.
5. `policy_iteration()` alternates evaluation and improvement until the policy
   stops changing, which yields the optimal policy for this finite MDP.
6. `policy_iteration_with_history()` does the same work but also records every
   intermediate snapshot so the learning process can be printed or visualized.
7. `main()` runs the procedure end-to-end and prints the value tables and
   policies for each iteration.

## Imports

In [43]:
from __future__ import annotations

from dataclasses import dataclass
from types import SimpleNamespace
from typing import Dict, List, Tuple

import numpy as np
import json
from pathlib import Path

## GridWorld Environment

This section defines the GridWorldConfig. It sets the dimensions ($3 \times 3$), the reward structure ($-1$ per step, $+10$ for the goal), and the stochasticity of the environment. Note that the agent only has an 80% chance of moving in the intended direction, with a 10% chance of "slipping" left or right.

In [44]:
UP = 0
RIGHT = 1
DOWN = 2
LEFT = 3

ACTION_NAMES = {
    UP: 'UP',
    RIGHT: 'RIGHT',
    DOWN: 'DOWN',
    LEFT: 'LEFT',
}

In [45]:
@dataclass(frozen=True)
class GridWorldConfig:
    """Immutable parameter bundle for the case-study grid world."""
    rows: int = 3
    cols: int = 3
    start_state: int = 0
    goal_state: int = 8
    step_reward: float = -1.0
    goal_reward: float = 10.0
    intended_move_prob: float = 0.8
    slip_left_prob: float = 0.1
    slip_right_prob: float = 0.1

In [46]:
class GridWorldCaseStudyEnv:
    """Tabular grid world with stochastic slips and a Gym-like API."""

    config: GridWorldConfig
    observation_space: SimpleNamespace
    action_space: SimpleNamespace
    P: Dict[int, Dict[int, List[Tuple[float, int, float, bool]]]]
    state: int
    rng: np.random.Generator

    def __init__(self, config: GridWorldConfig | None = None) -> None:
        self.config = config or GridWorldConfig()
        self.observation_space = SimpleNamespace(
            n=self.config.rows * self.config.cols
        )
        self.action_space = SimpleNamespace(n=4)
        self.P = self._build_transition_model()
        self.rng = np.random.default_rng()
        self.state = self.config.start_state

    def _build_transition_model(self):
        transitions = {}
        for state in range(self.observation_space.n):
            transitions[state] = {}
            for action in range(self.action_space.n):
                transitions[state][action] = self._transition_distribution(state, action)
        return transitions

    def _transition_distribution(self, state: int, action: int):
        if state == self.config.goal_state:
            return [(1.0, state, 0.0, True)]

        candidate_moves = [
            (action, self.config.intended_move_prob),
            ((action - 1) % self.action_space.n, self.config.slip_left_prob),
            ((action + 1) % self.action_space.n, self.config.slip_right_prob),
        ]

        next_state_probs = {}
        for move_action, prob in candidate_moves:
            if prob == 0.0:
                continue
            next_state = self._move(state, move_action)
            next_state_probs[next_state] = next_state_probs.get(next_state, 0.0) + prob

        transition_outcomes = []
        for next_state, prob in next_state_probs.items():
            done = next_state == self.config.goal_state
            reward = self.config.goal_reward if done else self.config.step_reward
            transition_outcomes.append((prob, next_state, reward, done))
        return transition_outcomes

    def _move(self, state: int, action: int) -> int:
        if state == self.config.goal_state:
            return state

        row, col = divmod(state, self.config.cols)
        next_row, next_col = row, col
        if action == UP:
            next_row = max(0, row - 1)
        elif action == RIGHT:
            next_col = min(self.config.cols - 1, col + 1)
        elif action == DOWN:
            next_row = min(self.config.rows - 1, row + 1)
        elif action == LEFT:
            next_col = max(0, col - 1)
        else:
            raise ValueError(f"Invalid action: {action}")

        return next_row * self.config.cols + next_col

    def reset(self):
        self.state = self.config.start_state
        return self.state, {}

    def step(self, action: int):
        outcomes = self.P[self.state][action]
        probabilities = [prob for prob, _, _, _ in outcomes]
        outcome_index = int(self.rng.choice(len(outcomes), p=probabilities))
        _, next_state, reward, done = outcomes[outcome_index]
        self.state = next_state
        return next_state, reward, done, False, {}

## Implementing Policy Evaluation

### Policy evaluation: compute $V^\pi$ for a fixed policy $\pi$.
This function computes the state-value function $V^\pi$ for a fixed policy $\pi$. It answers the question: "How much total reward do I expect to get from each state if I follow this specific policy?"

The Bellman Expectation Equation:

$$ V^\pi(s) = \sum_a \pi(a \mid s)\sum_{s',r} p(s',r \mid s,a)\left[r + \gamma V^\pi(s')\right]$$


Mechanism: The function performs "sweeps" across all states, updating the values until they stabilize (when the change $\Delta$ is smaller than $\theta$).

In [47]:
def policy_evaluation(env, policy, gamma=0.9, theta=1e-8, max_iterations=10000):
    """Iteratively evaluate a policy on the tabular grid-world MDP.

    Precondition:
    `env` exposes `observation_space.n` and a tabular transition model `P`,
    `policy` is a state-action probability matrix aligned with the environment,
    and `gamma` lies in a range that makes the Bellman expectation updates well
    behaved for the task.

    What happens:
    1. Initialize the value function `V` to zeros.
    2. Repeatedly sweep over all states. A sweep = one pass through every state,
       applying the Bellman equation once to each state. This is pure computation
       from the model — no agent interaction, no episodes.
    3. For each state, compute the Bellman expectation backup under `policy` by
       summing over actions and transition outcomes.
    4. Track `delta` — the largest absolute change to any state's value during
       this sweep. This tells us how much the values are still moving.
    5. Stop once `delta < theta`. `theta` is the convergence threshold — when no
       state's value changes by more than `theta` in an entire sweep, the values
       are close enough to the true V^pi and we stop.

    Postcondition:
    Returns the converged value-function estimate for the supplied policy.
    Raises `RuntimeError` if convergence is not reached within `max_iterations`.
    """
    num_states = env.observation_space.n
    V = np.zeros(num_states)
    
    for iteration in range(max_iterations):
        delta = 0.0
        
        # Sweep through all states
        new_V = np.copy(V)  # synchronous update: buffer all updates before applying

        for state in range(num_states):
            new_value = 0.0
            
            # Sum over all actions weighted by policy probabilities
            for action in range(env.action_space.n):
                action_prob = policy[state, action]
                
                # Sum over all transition outcomes for this state-action pair
                for prob, next_state, reward, done in env.P[state][action]:
                    # Bellman expectation equation with explicit terminal handling
                    # V(s) = sum_a pi(a|s) * sum_{s',r} p(s',r|s,a) * [r + gamma*V(s')]
                    if done:
                        new_value += action_prob * prob * reward
                    else:
                        new_value += action_prob * prob * (reward + gamma * V[next_state])
            
            new_V[state] = new_value
            delta = max(delta, abs(V[state] - new_value))
        
        V = new_V  # apply all updates at once (synchronous / tabular DP sweep)

        # Check for convergence
        if delta < theta:
            print(f"Policy Evaluation converged in {iteration + 1} sweeps")
            return V
    
    raise RuntimeError(f"policy_evaluation did not converge within max_iterations={max_iterations}")

### Action-Value (Q-Value) Computation
#### Implementing compute_q_from_v

While $V(s)$ tells us the value of a state, we need to know the value of specific actions to improve our policy. This function performs a one-step lookahead using the environment's transition model:

$$Q^\pi(s,a) = \sum_{s',r} P(s',r|s,a)[r + \gamma V^\pi(s')]$$

In [48]:
def compute_q_from_v(env, V, gamma=0.9):
    """Compute action values from a state-value function and the environment model.

    Precondition:
    `env` provides a tabular transition model `P`, `V` has one value per state,
    and `gamma` is the discount factor used for the one-step lookahead.

    What happens:
    1. Allocate a zero-filled Q-table.
    2. For every state and action, enumerate all transition outcomes.
    3. Apply the Bellman one-step lookahead backup using `V[next_state]`.
    4. Sum the expected discounted return into `Q[s, a]`.

    Postcondition:
    Returns a full state-action value table consistent with `V` and `env.P`.
    """
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    Q = np.zeros((num_states, num_actions))
    
    for state in range(num_states):
        for action in range(num_actions):
            # Sum over all transition outcomes for this state-action pair
            for prob, next_state, reward, done in env.P[state][action]:
                # Bellman one-step lookahead with explicit terminal handling
                # Q(s,a) = sum_{s',r} P(s',r|s,a) * [r + gamma * V(s')]
                if done:
                    Q[state, action] += prob * reward
                else:
                    Q[state, action] += prob * (reward + gamma * V[next_state])
    
    return Q

In [49]:
def format_q_tie_details(Q):
    """Describe states whose greedy action set contains ties."""
    lines = []
    for state, row in enumerate(np.asarray(Q, dtype=float)):
        best_value = row.max()
        tied_actions = np.flatnonzero(np.isclose(row, best_value))
        if len(tied_actions) > 1:
            row_values = ', '.join(f'{value:.15f}' for value in row)
            action_names = ', '.join(ACTION_NAMES[int(action)] for action in tied_actions)
            lines.append(
                f'state {state}: Q=[{row_values}] tied_best_actions=[{action_names}]'
            )
    return '\n'.join(lines) if lines else 'No tied greedy actions.'

## Implementing Policy Improvement

This step makes the agent "smarter." It takes the values calculated in the previous step and updates the policy to favor actions with higher Q-values.  
$\epsilon$-Soft Strategy: To prevent the agent from getting stuck and to ensure it keeps exploring, we use an $\epsilon$-greedy approach.  
Most of the probability ($1 - \epsilon$) is given to the greedy (best) actions, while a small amount ($\epsilon$) is spread across all possible actions.

In [50]:
def policy_improvement(env, V, gamma=0.9, epsilon=0.1):
    """Build an epsilon-soft improved policy from the current value function.

    Precondition:
    `env` and `V` are compatible, and `epsilon` is chosen so that a valid
    epsilon-soft policy can be formed.

    What happens:
    1. Compute the Q-table implied by `V`.
    2. Start from a uniform exploration floor of `epsilon / num_actions`.
    3. Identify all greedy actions in each state, preserving ties.
    4. Split the remaining `1 - epsilon` probability mass evenly across the
       greedy action set.

    Postcondition:
    Returns the improved epsilon-soft policy together with the Q-table used to
    construct it.
    """
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    
    # Compute Q-table from V
    Q = compute_q_from_v(env, V, gamma=gamma)
    
    # Initialize policy with uniform exploration floor
    policy = np.full((num_states, num_actions), epsilon / num_actions)
    
    # For each state, add remaining probability mass to greedy actions
    for state in range(num_states):
        # Find all actions tied for the maximum Q-value
        q_values = Q[state]
        max_q_value = q_values.max()
        greedy_actions = np.flatnonzero(np.isclose(q_values, max_q_value))
        
        # Distribute remaining (1 - epsilon) probability equally among greedy actions
        improvement_prob_per_action = (1.0 - epsilon) / len(greedy_actions)
        for action in greedy_actions:
            policy[state, action] += improvement_prob_per_action
    
    return policy, Q

## Policy Iteration Loop

This is the main orchestration loop. Policy Iteration works by alternating between two phases:  
1. Evaluation: Calculating the value of the current policy.
2. Improvement: Updating the policy to be greedy with respect to those values.  
The process repeats until the policy becomes stable (no longer changes), meaning we have found the Optimal Policy ($\pi^*$).

In [51]:
def policy_iteration_with_history(env, gamma=0.9, theta=1e-8, epsilon=0.1, max_policy_iterations=1000):
    """Run policy iteration and retain every intermediate policy-improvement step."""
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    policy = np.ones((num_states, num_actions)) / num_actions
    history = []
    iteration = 0
    
    for _ in range(max_policy_iterations):
        V = policy_evaluation(env, policy, gamma=gamma, theta=theta)
        new_policy, Q = policy_improvement(env, V, gamma=gamma, epsilon=epsilon)
        stable = np.array_equal(new_policy.argmax(axis=1), policy.argmax(axis=1))
        
        history.append({
            'iteration': iteration,
            'policy': new_policy.copy(),
            'V': V.copy(),
            'Q': Q.copy(),
            'stable': stable,
        })
        
        if stable:
            print(f"\nPolicy Iteration converged in {iteration + 1} iterations")
            return new_policy, V, Q, history
        
        policy = new_policy
        iteration += 1
    
    raise RuntimeError(
        f'policy_iteration_with_history did not converge within max_policy_iterations={max_policy_iterations}'
    )

## Visualization Helpers

In [52]:
def _format_action_grid(rows) -> str:
    lines = []
    cols = GridWorldConfig().cols
    separator = '+---+---+---+'
    lines.append(separator)
    for idx in range(0, len(rows), cols):
        lines.append('|' + '|'.join(rows[idx: idx + cols]) + '|')
        lines.append(separator)
    return '\n'.join(lines)


def format_policy(policy) -> str:
    """Render a policy as a text grid of greedy action arrows."""
    arrows = {UP: '^', RIGHT: '>', DOWN: 'v', LEFT: '<'}
    rows = []
    for state in range(policy.shape[0]):
        if state == GridWorldConfig().goal_state:
            rows.append(' G ')
        else:
            row = np.asarray(policy[state], dtype=float)
            best_value = row.max()
            best_actions = np.flatnonzero(np.isclose(row, best_value))
            cell = ''.join(arrows[action] for action in best_actions)
            rows.append(f'{cell:^3}')
    return _format_action_grid(rows)


def format_greedy_actions(Q) -> str:
    """Render the greedy action set implied by a Q-table as a text grid."""
    arrows = {UP: '^', RIGHT: '>', DOWN: 'v', LEFT: '<'}
    rows = []
    q_values = np.asarray(Q, dtype=float)
    for state in range(q_values.shape[0]):
        if state == GridWorldConfig().goal_state:
            rows.append(' G ')
        else:
            row = q_values[state]
            best_value = row.max()
            best_actions = np.flatnonzero(np.isclose(row, best_value))
            cell = ''.join(arrows[int(action)] for action in best_actions)
            rows.append(f'{cell:^3}')
    return _format_action_grid(rows)

## Run The Case Study

In [53]:
env = GridWorldCaseStudyEnv()
epsilon = 0.1
policy, V, Q, history = policy_iteration_with_history(env, epsilon=epsilon)

np.set_printoptions(precision=3, suppress=True)
print('Dynamic Programming Snapshots')
for snapshot in history:
    print(f"\nIteration {snapshot['iteration']} (stable={snapshot['stable']})")
    print('V(s)')
    print(snapshot['V'].reshape(3, 3))
    print('Q(s,a)')
    print(snapshot['Q'])
    print('Full-precision tied greedy Q rows')
    print(format_q_tie_details(snapshot['Q']))
    print('Greedy actions from Q (ties shown)')
    print(format_greedy_actions(snapshot['Q']))
    print(f"Epsilon-greedy policy (epsilon={epsilon})")
    print(format_policy(snapshot['policy']))

print('Optimal state values V*')
print(V.reshape(3, 3))
print('\nOptimal action values Q*')
print(Q)
print('\nFull-precision tied greedy Q rows')
print(format_q_tie_details(Q))
print('\nGreedy actions from Q* (ties shown)')
print(format_greedy_actions(Q))
print(f"\nEpsilon-greedy policy from DP (epsilon={epsilon})")
print(format_policy(policy))

Policy Evaluation converged in 121 sweeps
Policy Evaluation converged in 36 sweeps

Policy Iteration converged in 2 iterations
Dynamic Programming Snapshots

Iteration 0 (stable=False)
V(s)
[[-5.922 -5.016 -3.767]
 [-5.016 -3.144  0.251]
 [-3.767  0.251  0.   ]]
Q(s,a)
[[-6.249 -5.596 -5.596 -6.249]
 [-5.484 -4.447 -4.136 -5.999]
 [-4.503 -4.029 -1.61  -4.928]
 [-5.999 -4.136 -4.447 -5.484]
 [-5.041 -1.248 -1.248 -5.041]
 [-3.973 -0.058  7.54  -2.503]
 [-4.928 -1.61  -4.029 -4.503]
 [-2.503  7.54  -0.058 -3.973]
 [ 0.     0.     0.     0.   ]]
Full-precision tied greedy Q rows
state 0: Q=[-6.248688522349744, -5.596286538399133, -5.596286538399132, -6.248688522349744] tied_best_actions=[RIGHT, DOWN]
state 4: Q=[-5.040632584489120, -1.247802937159619, -1.247802937159618, -5.040632584489118] tied_best_actions=[RIGHT, DOWN]
state 8: Q=[0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000] tied_best_actions=[UP, RIGHT, DOWN, LEFT]
Greedy actions from Q (ties shown)
+--

## Final Output

In [54]:
print('\n' + '='*70)
print('FINAL DYNAMIC PROGRAMMING OUTPUT')
print('='*70)
print('Stable policy:', history[-1]['stable'])
print('\nFinal state values V*')
print(V.reshape(3, 3))
print('\nFinal action values Q*')
print(Q)
print('\nFull-precision tied greedy Q rows')
print(format_q_tie_details(Q))
print('\nGreedy actions from Q* (ties shown)')
print(format_greedy_actions(Q))
print(f"\nFinal epsilon-greedy policy (epsilon={epsilon})")
print(format_policy(policy))


FINAL DYNAMIC PROGRAMMING OUTPUT
Stable policy: True

Final state values V*
[[2.71  4.352 6.22 ]
 [4.352 6.458 8.927]
 [6.22  8.927 0.   ]]

Final action values Q*
[[1.587 2.769 2.769 1.587]
 [2.937 4.451 4.454 1.924]
 [4.43  4.842 6.379 3.497]
 [1.924 4.454 4.451 2.937]
 [3.329 6.623 6.623 3.329]
 [4.863 7.087 9.185 5.31 ]
 [3.497 6.379 4.842 4.43 ]
 [5.31  9.185 7.087 4.863]
 [0.    0.    0.    0.   ]]

Full-precision tied greedy Q rows
state 0: Q=[1.587030306351329, 2.769380340739469, 2.769380340739469, 1.587030306351329] tied_best_actions=[RIGHT, DOWN]
state 4: Q=[3.328918323594937, 6.622872152050545, 6.622872152050546, 3.328918323594937] tied_best_actions=[RIGHT, DOWN]
state 8: Q=[0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000] tied_best_actions=[UP, RIGHT, DOWN, LEFT]

Greedy actions from Q* (ties shown)
+---+---+---+
|>v | v | v |
+---+---+---+
| > |>v | v |
+---+---+---+
| > | > | G |
+---+---+---+

Final epsilon-greedy policy (epsilon=0.1)
+---+---

### Analysis and Justification of Output

The algorithm successfully transitions from a random strategy to an optimal policy in just **2 iterations**. Below is a detailed breakdown of the results.

---

#### A. Convergence Speed

**Observation:**  
Policy Iteration converged in **2 iterations**.

**Justification:**  
In small, discrete state spaces like a $3 \times 3$ grid, the number of possible deterministic policies is finite (approximately $4^8$ for the non-terminal states).  

Policy Iteration is highly efficient in such environments because:
- It evaluates and improves policies directly
- The policy typically stabilizes **before** state values $V(s)$ fully converge in precision

---

#### B. Transition from Iteration 0 to Iteration 1

#### Iteration 0 (Random Start)
- State values $V(s)$ are **negative** (e.g., State 0 ≈ -5.922)
- The agent follows a **uniform random policy**

**Interpretation:**
- The agent wanders randomly
- It accumulates a step penalty (e.g., $-1$ per move)
- Reaching the goal is rare and accidental

---

##### Iteration 1 (After Policy Improvement)
- State values become **positive** (e.g., State 0 ≈ 2.71)

**Interpretation:**
- The agent has discovered a **directed path to the goal**
- The terminal reward ($+10$) outweighs cumulative step penalties
- Value propagation from the goal is now effective

---

#### C. Value Gradient and the Goal

Final optimal state values:

$$
V^* =
\begin{bmatrix}
2.710 & 4.352 & 6.220 \\
4.352 & 6.458 & 8.927 \\
6.220 & 8.927 & 0.000
\end{bmatrix}
$$

**Justification:**
- Values increase as the agent gets closer to the goal (State 8)
- States 5 and 7 (value ≈ 8.927) are most valuable since they are one step away

**Why not exactly 9.0?**
- Discount factor $\gamma = 0.9$ reduces future reward
- Environment stochasticity (e.g., 10% slip probability) lowers expected returns

---

#### D. Understanding Ties in the Policy

**Observation:**  
Some states have **multiple optimal actions**, e.g.:
- State 0 → RIGHT and DOWN
- State 4 → RIGHT and DOWN

**Justification:**
- The grid is **symmetric**
- Multiple paths have identical **Manhattan distance** to the goal

Example:
- RIGHT → DOWN ≡ DOWN → RIGHT

**Interpretation:**
- Both actions yield identical $Q(s,a)$ values
- The policy correctly represents this using combined arrows (e.g., `>v`)

---

#### E. Impact of Epsilon ($\epsilon = 0.1$)

**Observation:**  
The epsilon-greedy policy visualization looks identical to the greedy policy.

**Justification:**
- The visualization shows only **greedy actions (argmax)**
- The actual policy is **$\epsilon$-soft**:
  - 90% probability → best action
  - 10% probability → random exploration

**Interpretation:**
- Ensures robustness against environment stochasticity (e.g., slipping)
- Prevents over-reliance on deterministic transitions

---

#### Summary Table of Final Results

| Metric        | Value           | Meaning |
|--------------|----------------|--------|
| Convergence  | 2 Iterations   | Policy stabilized quickly |
| Max $V^*$    | 8.927          | Highest value (adjacent to goal) |
| Min $V^*$    | 2.710          | Start state value (cost + risk) |
| Policy Type  | $\epsilon$-soft | 90% exploitation, 10% exploration |

---

#### Final Intuition

This process is like learning to navigate a maze:

- **Iteration 0:** Random wandering with penalties  
- **Iteration 1:** Clear direction toward the goal  
- **Final Policy:** Always move along the highest-value path  

Dynamic Programming works by **propagating value backward from the goal**, refining decisions at each step.

## Export Policies To JSON

In [55]:
transition_model = {
    str(state): {
        ACTION_NAMES[action].upper(): [
            {
                'probability': float(prob),
                'next_state': int(next_state),
                'reward': float(reward),
                'done': bool(done),
            }
            for prob, next_state, reward, done in outcomes
        ]
        for action, outcomes in actions.items()
    }
    for state, actions in env.P.items()
}


payload = {
    'epsilon': epsilon,
    'action_order': [ACTION_NAMES[i].upper() for i in range(4)],
    'final': {
        'stable': bool(history[-1]['stable']) if history else True,
        'policy_matrix': policy.tolist(),
        'V': V.tolist(),
        'Q': Q.tolist(),
        'transition_model': transition_model,
    },
}

output_path = Path('dynamic_programming_policies.json')
output_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f"Output saved to: {output_path.resolve()}")

Output saved to: D:\PycharmProjects\Mini\dynamic_programming_policies.json
